# Working with Claude API
***

## Making a request
Messages represent the conversation between you and Claude, similar to a chat application. There are two types of messages:
 - **User messages** - Content you want to send to Claude (written by humans)
 - **Assistant messages** - Responses that Claude has generated
Each message is a dictionary with a role (either "user" or "assistant") and content (the actual text).

In [2]:
# Installing necessary dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Adding the env variables & api client

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
# Making a request to the API

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]
)

message.content[0].text

'Quantum computing is a revolutionary computing paradigm that uses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain complex problems exponentially faster than classical computers.'

## Multi-Turn Conversations

When working with the Anthropic API and Claude, there's a crucial concept you need to understand: **Claude doesn't store any of your conversation history.** Each request you make is completely independent, with no memory of previous exchanges.

Therefore, to have a conversation, you need to manually maintain a list of messages in your code, and provide that list of messages with each request

We can do this with a variety of helper functions

In [10]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

In [11]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

## Building a simiple Chatbot

Use the helper functions above and create a simple chatbot using user input

In [13]:
# Make a list of the messages
messages = []

# Make a while loop to keep the conversation going until the user types "exit"
while True:
    user_input = input("> ")
    if user_input == "exit":
        break
    print("> ", user_input)

    # Add user input to the list of messages
    add_user_message(messages, user_input)
    # Call Claude with the 'chat' function
    claude_response = chat(messages)
    # Add generated text to the list of messages
    add_assistant_message(messages, claude_response)
    # Print generated response
    print("---")
    print("> ", claude_response)
    print("---")


>  What's 1+1?
---
>  1 + 1 = 2
---
>  Add 1 to that last answer
---
>  2 + 1 = 3
---


## System Prompts

**System prompts** are a way to customize how Claude responds to the user input. Instead of getting generic answers, system prompts can shape Claude's tone, style, and approach to match your specific use case. This can help tailor Claude to your preferences, setting do's and don'ts for Claude's answers

System prompts provide Claude with **guidance** on how to respond. You define them as plain strings and pass them into the create function call. The key benefits are:
 - System prompts provide Claude guidance on how to respond. 
 - Claude will try to respond in the same way someone in the specified role would respond
 - Helps keep Claude on task

In [ ]:
system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

client.messages.create(
    model=model,
    messages=messages,
    max_tokens=1000,
    system=system_prompt
)

Without a system prompt, Claude gives the full step-by-step solution of math problems in the usecase above. This might be helpful but it does not encourage the user to think about the problem themselves. With the system_prompt of being a math tutor, Claude knows what it needs to show and hide in terms of its responses, and thus gives more tailored answers

In [14]:
# Rather than creating a bunch of hard-coding system prompts, you can make your chat function
# more flexible by allowing it to take an optional system prompt argument.

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

messages = []
add_user_message(messages, "Write a python function that checks a string for duplicate characters,")
answer = chat(messages, system="You are a Python engineer that writes concise code.")

answer

'Here\'s a concise Python function to check for duplicate characters:\n\n```python\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n```\n\nThis function works by:\n1. Converting the string to a set (which removes duplicates)\n2. Comparing the lengths - if they differ, duplicates existed\n\nExample usage:\n```python\nprint(has_duplicates("hello"))    # True (duplicate \'l\')\nprint(has_duplicates("world"))    # False (no duplicates)\nprint(has_duplicates(""))         # False (empty string)\n```\n\nIf you need to know which characters are duplicated:\n```python\ndef find_duplicates(s):\n    seen = set()\n    return {char for char in s if char in seen or seen.add(char)}\n```'

## Temperature

**Temperature** is a powerful parameter that controls how predictable or creative Claude's responses will be. 

When you send a prompt, Claude goes through 3 key steps:
1. **Tokenization**: Breaking your input into smaller chunks
2. **Prediction**: Calculating probabilities for possible next words
3. **Sampling**: Choosing a token based on those probabilities

Temperature is a decimal value between 0 and 1 that directly influences these selection probabilities. It's like adjusting the "creativity dial" on Claude's responses. 
 - **Low temperatures (near 0)**, Claude becomes deterministic. Always picks the highest probability token
 - **High temperatures (near 1)**, Claude distributes probability evenly, leading to more varied/creative outputs

In general, you want to use temperature in these cases:
| Low (0.0-0.3) | Middle (0.4-0.7) | High (0.8-1.0) |
| --- | ------ | ---- |
| factual responses | summarization | brainstorming |
| coding assistance | educational content | creative writing |
| data extraction | problem-solving | marketing content |
| content moderation | creative writing with restraint | joke generation| 

In [17]:
# Modifying chat function to take temperature as an argument. 

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

messages = []
add_user_message(messages, "Generate a movie idea")
# Low temperature - more predictable
answer1 = chat(messages, temperature=0.0)
print(answer1)

# High temperature - more creative  
answer2 = chat(messages, temperature=1.0)
print(answer2)

**Title: "The Memory Thief"**

**Genre:** Sci-fi Thriller

**Logline:** In 2045, a black-market memory extractor who helps people erase traumatic experiences discovers that someone is stealing and weaponizing the memories she's collected, forcing her to dive into a dangerous world of corporate espionage and underground resistance.

**Plot Summary:**
Maya Chen operates an illegal clinic where she uses experimental technology to extract and destroy painful memories—helping war veterans forget PTSD, abuse survivors move on, and addicts overcome triggers. She stores these extracted memories in a secure digital vault, believing them safely destroyed.

When clients start dying under mysterious circumstances, Maya realizes someone has been stealing the memories from her vault and selling them to Nexus Corp, a tech giant developing empathy-based AI for military applications. The stolen trauma memories are being used to create emotionally manipulative weapons and propaganda.

Maya must team up 

**Temperature** does not guarantee different outputs -- it just changes the probability of getting them. Even at high temperatures, Claude might occasionally produce similar responses. 

## Response Streaming

Responses can take 10-30 seconds to generate, leaving the users waiting. In order to streamline this process, **response streaming** lets users see the response being generated chunk by chunk, creating a more responsive feel.

Currently, we are sending a user message to Claude and waits for the **complete** response before sending anything back to the client. This creates the wait time. Through streaming, Claude immediately sends back an **initial response** indicating that it successfully received your message. Then, you receive a **series of events** containing a small piece of the response. Your server can forward these text chunks to your client application as they arrive, allowing users to see the response building up word by word.

When you enable streaming, Claude sends back several types of events:
 - **MessageStart**: A new message is being sent
 - **ContentBlockStart**: Start of a new block containing text, tool use, or other content
 - **ContentBlockDelta**: Chunks of the actual generated text
 - **ContentBlockStop**: The current content block has been completed
 - **MessageDelta**: The current message is complete
 - **MessageStop**: End of information about the current message

The **ContentBlockDelta** events contain the acutal generated text that you want to display to the users. 

In [18]:
# Set stream=True to enable response streaing

messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_012rZhHDmpTAoMWRptXcNuDb', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='UserVault is a cloud-based customer relationship management database that stores detailed user profiles, purchase histories, and behavioral', type='text_delta'), index=0, type='content_block_delta')

In [21]:
# Simplified, filtering everything except for content

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        #print(text, end="")
        pass
    final_message = stream.get_final_message()
print(final_message.content[0].text)

The Global Interdimensional Pet Registry (GIPR) is a comprehensive database containing behavioral profiles, dimensional origin coordinates, and care instructions for over 2.7 million registered pets from parallel universes, including quantum cats, phase-shifting dogs, and time-dilated hamsters.


## Structured Data

When you want Claude to generate code or data, Claude often generates 'helpful' and 'explanatory' text around the content. This is great in some cases, but sometimes you just need to raw data. 

In order to fix this, you can combine assistant message prefilling with stop preferences to get exactly the content you want. This technique works by: 
1. The user message tells Claude what to generate
2. The prefilled assistant message makes Claude think it already started a markdown code block
3. Claude continues by writing just the JSON content (could be any code)
4. When Claude tries to close the code block with ```, the stop sequence immediately ends generation

The result is a clean code with no extra text.

In [27]:
messages = []

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    message = client.messages.create(**params)
    return message.content[0].text

add_user_message(messages, "Generate a very short event bridge rule as json with aws")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
print(text)


{
  "Name": "S3UploadRule",
  "EventPattern": {
    "source": ["aws.s3"],
    "detail-type": ["Object Created"],
    "detail": {
      "bucket": {
        "name": ["my-bucket"]
      }
    }
  },
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessUpload"
    }
  ]
}



In [ ]:
# Exercise: Generate three different sample AWS CLI commands. Each should be very short.

message = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(message, prompt)
add_assistant_message(message, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(message, stop_sequences=["```"])
text.strip()

'aws s3 ls\naws ec2 describe-instances\naws iam list-users'